# Attack 3 — Context-Aware Substitution (BERT)

Reproduction of the context-aware substitution attack from Gloaguen et al. (ICLR 2026),
Figure 3 (left). MarkLLM `ContextAwareSynonymSubstitution` (unchanged) with
`bert-large-uncased`, as in `scripts/run_robustness_eval.py`. Ratios: 0.1–0.5.

Note on the implementation: WordNet is only used to select which words are replaced.
Each selected word is masked and replaced by BERT's top-1 prediction, which is not
checked to be a synonym and is often the original word itself (lowercased, since the
model is uncased). We therefore also count the words that actually change.

Runs locally on Mac (Apple GPU via MPS if available, otherwise CPU).

## 1. Setup

In [1]:
import random
import sys
import time
from importlib.metadata import version
from pathlib import Path

import nltk
import numpy as np
import torch
from transformers import BertForMaskedLM, BertTokenizer


def find_project_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / "diffusion-lm-watermark").exists():
            return folder
    raise FileNotFoundError("Project root not found.")


ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))
from detection import ALPHA, detect, load_detector, load_jsonl, save_jsonl

SEED = 42
nltk.download("wordnet", quiet=True)

wm = load_jsonl(ROOT / "results" / "Cleaned Data" / "watermarked_600.jsonl")
baseline = load_jsonl(ROOT / "results" / "attacks" / "baseline_original.jsonl")
assert len(wm) == len(baseline) == 600
assert all(
    row["prompt"] == base["prompt"]
    for row, base in zip(wm, baseline)
)
OUT_DIR = ROOT / "results" / "attacks" / "context_aware_substitution"
tokenizer, detector = load_detector(ROOT)

# BERT runs on the Apple GPU (MPS) if available; the watermark detector stays on CPU
BERT_DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
bert_tokenizer = BertTokenizer.from_pretrained("bert-large-uncased")
bert_model = BertForMaskedLM.from_pretrained("bert-large-uncased").to(BERT_DEVICE).eval()

print("Watermarked records:", len(wm), "| baseline records:", len(baseline))
print("BERT device:", BERT_DEVICE)
for pkg in ["markllm", "torch", "transformers"]:
    print(f"{pkg}: {version(pkg)}")

/Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1118 03:37:58.413000 83338 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 394/394 [00:00<00:00, 7428.22it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-large-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Watermarked records: 600 | baseline records: 600
BERT device: mps
markllm: 0.1.5
torch: 2.15.0
transformers: 5.22.1


## 2. Pilot: 3 texts, timing and actual changes

Ratios 0.1 and 0.3 on 3 texts (one per dataset). Each word keeps its position (BERT
predicts one token per masked word), so original and attacked texts can be compared
word by word:
- `changed_raw`: words that differ at all (including lowercasing / lost punctuation),
- `changed_real`: words that differ after lowercasing and stripping punctuation.
We also time each attack to estimate the cost of the full run.

In [2]:
import string
from markllm.evaluation.tools.text_editor import ContextAwareSynonymSubstitution
from nltk.corpus import wordnet


def normalize(word: str) -> str:
    return word.lower().strip(string.punctuation)


def count_changes(original: str, attacked: str) -> dict:
    """Word-by-word comparison (positions are preserved by this attack)."""
    o, a = original.split(), attacked.split()
    if len(o) != len(a):              # should not happen; flagged for inspection
        return {"aligned": False, "changed_raw": None, "changed_real": None}
    return {
        "aligned": True,
        "changed_raw": sum(x != y for x, y in zip(o, a)),
        "changed_real": sum(normalize(x) != normalize(y) for x, y in zip(o, a)),
    }


def n_targeted(text: str, ratio: float) -> int:
    """Replicate MarkLLM's selection: words with at least one WordNet lemma."""
    words = text.split()
    n_repl = sum(1 for w in words if wordnet.synsets(w))
    return int(min(ratio, n_repl / len(words)) * len(words))


PILOT_IDX = [0, 200, 400]
PILOT_RATIOS = [0.1, 0.3]
editors = {
    r: ContextAwareSynonymSubstitution(
        ratio=r, tokenizer=bert_tokenizer, model=bert_model, device=BERT_DEVICE
    )
    for r in PILOT_RATIOS
}

for idx in PILOT_IDX:
    original = wm[idx]["completion"]
    base = baseline[idx]
    print("=" * 100)
    print(f"Text {idx} | {base['dataset']} | original: p={base['p_value']:.2e}  "
          f"green={base['green_fraction']:.3f}  words={len(original.split())}")
    for ratio in PILOT_RATIOS:
        random.seed(SEED + idx)
        t0 = time.time()
        attacked = editors[ratio].edit(original)
        elapsed = time.time() - t0
        res = detect(attacked, tokenizer, detector)
        ch = count_changes(original, attacked)
        print(f"\n  ratio {ratio:.1f} | targeted {n_targeted(original, ratio)} | "
              f"changed raw {ch['changed_raw']} | changed real {ch['changed_real']} | "
              f"{elapsed:.1f}s")
        print(f"  p={res['p_value']:.2e}  green={res['green_fraction']:.3f}  "
              f"detected={res['detected']}")
        print("  ATTACKED:", attacked[:300], "...")

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/paulmicheli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/paulmicheli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Text 0 | 2-1_longform_qa | original: p=2.81e-15  green=0.480  words=250

  ratio 0.1 | targeted 25 | changed raw 7 | changed real 7 | 5.5s
  p=2.66e-14  green=0.474  detected=True
  ATTACKED: The case of the Soviet Union is when one of the largest superpowers, or a big super-strong robot that had been fighting together for many years, suddenly falls apart all by itself. Sometimes, a robot can fall apart because they don't work together anymore, or because they've gotten too far away that ...

  ratio 0.3 | targeted 75 | changed raw 26 | changed real 25 | 9.9s
  p=8.77e-11  green=0.438  detected=True
  ATTACKED: The case of the Soviet Union is when one of the two superpowers, or a big super-strong robot that has been working together for many years, suddenly falls apart all by itself. Sometimes, a robot can fall apart because they don't work together anymore, or because they've moved too far away that it's  ...
Text 200 | 2-2_finance_qa | original: p=9.39e-10  green=0.444  words=252

  


### Pilot Observations

BERT changes fewer words than the requested substitution ratio.
Some selected words are predicted as the original word and remain unchanged.

The examples also show that a context-aware substitution can change
the meaning of a sentence, even when the resulting text is fluent.

The full evaluation will measure the attack success rate across
all 600 responses.



## 3. Full Run (Resumable)

We apply BERT context-aware substitution to all 600 watermarked
responses at ratios from 0.1 to 0.5.

Each ratio is processed separately. If its output file already
exists, the experiment is skipped.

BERT substitutions are performed sequentially because each
prediction uses the text with previous substitutions.

For each response, we save the attacked text, detection results,
number of targeted and changed words, and execution time.

Results are saved in separate JSONL files for each ratio.


In [ ]:
MARKLLM_VERSION = version("markllm")


def run_ratio(ratio: float) -> None:
    out_path = OUT_DIR / f"context_aware_substitution_r{ratio:.1f}.jsonl"
    if out_path.exists():
        print(f"ratio {ratio:.1f}: already done, skipped")
        return

    editor = ContextAwareSynonymSubstitution(
        ratio=ratio, tokenizer=bert_tokenizer, model=bert_model, device=BERT_DEVICE
    )
    records, t_start = [], time.time()
    for idx, row in enumerate(wm):
        original = row["completion"]
        random.seed(SEED + idx)
        t0 = time.time()
        attacked = editor.edit(original)
        attack_seconds = time.time() - t0
        res = detect(attacked, tokenizer, detector)
        base = baseline[idx]
        records.append({
            "idx": idx,
            "dataset": base["dataset"],
            "attack": "context_aware_substitution",
            "ratio": ratio,
            "seed": SEED + idx,
            "markllm_version": MARKLLM_VERSION,
            "bert_model": "bert-large-uncased",
            "bert_device": BERT_DEVICE,
            "prompt": row["prompt"],
            "original": original,
            "attacked": attacked,
            "n_words_original": len(original.split()),
            "n_targeted": n_targeted(original, ratio),
            **count_changes(original, attacked),
            "attack_seconds": attack_seconds,
            "original_p_value": base["p_value"],
            "original_detected": base["detected"],
            "original_green_fraction": base["green_fraction"],
            **res,
        })
        if (idx + 1) % 50 == 0:
            minutes = (time.time() - t_start) / 60
            print(f"  ratio {ratio:.1f}: {idx + 1}/600 texts, {minutes:.1f} min elapsed")

    save_jsonl(records, out_path)
    n_det = sum(r["detected"] for r in records if r["detected"] is not None)
    print(f"ratio {ratio:.1f}: detected {n_det}/600 | "
          f"total {(time.time() - t_start) / 60:.1f} min")


for ratio in [0.1, 0.2, 0.3, 0.4, 0.5]: 
    run_ratio(ratio)

ratio 0.1: already done, skipped
ratio 0.2: already done, skipped
ratio 0.3: already done, skipped
ratio 0.4: already done, skipped


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/paulmicheli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


  ratio 0.5: 50/600 texts, 13.1 min elapsed
  ratio 0.5: 100/600 texts, 27.1 min elapsed
  ratio 0.5: 150/600 texts, 40.4 min elapsed
  ratio 0.5: 200/600 texts, 53.5 min elapsed
  ratio 0.5: 250/600 texts, 65.9 min elapsed
  ratio 0.5: 300/600 texts, 92.1 min elapsed
  ratio 0.5: 350/600 texts, 105.1 min elapsed
  ratio 0.5: 400/600 texts, 119.0 min elapsed
  ratio 0.5: 450/600 texts, 130.8 min elapsed
  ratio 0.5: 500/600 texts, 145.3 min elapsed
  ratio 0.5: 550/600 texts, 156.4 min elapsed
  ratio 0.5: 600/600 texts, 168.5 min elapsed
Saved 600 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/context_aware_substitution/context_aware_substitution_r0.5.jsonl
ratio 0.5: detected 453/600 | total 168.5 min



## 4. Results and Analysis

We evaluate BERT substitution using the same detection rate
and Attack Success Rate (ASR) as in the previous attacks.

We also measure the proportion of words targeted for substitution
and the proportion that actually change. Differences in capitalization
and punctuation are ignored when counting real changes.

The attack success rate increases from 2.2% at ratio 0.1 to
22.3% at ratio 0.5.

However, only about 32% of the targeted words are actually
changed. BERT often predicts the original word, which reduces
the effective strength of the attack.

All 600 responses remain aligned, allowing a direct word-by-word
comparison.


In [14]:
from scipy.stats import binomtest

def wilson(k, n):
    ci = binomtest(k, n).proportion_ci(method="wilson")
    return f"[{ci.low:.1%}, {ci.high:.1%}]"

for path in sorted(OUT_DIR.glob("context_aware_substitution_r*.jsonl")):
    recs = load_jsonl(path)
    ratio = recs[0]["ratio"]
    n_det = sum(r["detected"] for r in recs)
    init = [r for r in recs if r["original_detected"]]
    n_removed = sum(not r["detected"] for r in init)
    aligned = [r for r in recs if r["aligned"]]
    targeted = sum(r["n_targeted"] for r in aligned)
    real = sum(r["changed_real"] for r in aligned)
    raw = sum(r["changed_raw"] for r in aligned)
    words = sum(r["n_words_original"] for r in aligned)
    print(f"ratio {ratio:.1f}")
    print(f"  detection rate   {n_det/600:6.1%} {wilson(n_det, 600)}")
    print(f"  ASR (on {len(init)})    {n_removed/len(init):6.1%} {wilson(n_removed, len(init))}")
    print(f"  mean green       {np.mean([r['green_fraction'] for r in recs]):.3f}")
    print(f"  words targeted   {targeted/words:6.1%} of all words")
    print(f"  changed (raw)    {raw/words:6.1%}   changed (real) {real/words:6.1%}")
    print(f"  real / targeted  {real/targeted:6.1%}")
    print(f"  misaligned texts {len(recs) - len(aligned)}")

ratio 0.1
  detection rate    95.0% [93.0%, 96.5%]
  ASR (on 583)      2.2% [1.3%, 3.8%]
  mean green       0.429
  words targeted     9.8% of all words
  changed (raw)      3.7%   changed (real)   3.1%
  real / targeted   32.1%
  misaligned texts 0
ratio 0.2
  detection rate    92.0% [89.6%, 93.9%]
  ASR (on 583)      5.3% [3.8%, 7.4%]
  mean green       0.411
  words targeted    19.8% of all words
  changed (raw)      7.4%   changed (real)   6.3%
  real / targeted   31.6%
  misaligned texts 0
ratio 0.3
  detection rate    89.7% [87.0%, 91.9%]
  ASR (on 583)      7.7% [5.8%, 10.2%]
  mean green       0.395
  words targeted    29.8% of all words
  changed (raw)     11.2%   changed (real)   9.5%
  real / targeted   31.8%
  misaligned texts 0
ratio 0.4
  detection rate    83.3% [80.1%, 86.1%]
  ASR (on 583)     14.2% [11.6%, 17.3%]
  mean green       0.379
  words targeted    39.8% of all words
  changed (raw)     14.9%   changed (real)  12.6%
  real / targeted   31.7%
  misaligned texts


## 5. Conclusion

BERT substitution produces fewer actual word changes than
the requested substitution ratio suggests.

This is important when comparing its effectiveness with
word deletion and WordNet substitution.

Although the replacements are context-aware, they do not
always preserve the original meaning. Text quality was not
evaluated quantitatively.
